# GAT - Graph Attention Network

GCN aggregates neighbors equally, every neighbor contributes the same weight.
GAT improves on this by learning which neighbors are more relevant through an 
attention mechanism, similar to transformers in NLP.

The hypothesis: fraudulent transactions may cluster with other suspicious nodes.
Attention should learn to weight those neighbors more heavily.

In [1]:
import torch
import torch.nn.functional as F
from torch_geometric.datasets import EllipticBitcoinDataset
from torch_geometric.nn import GATConv
from sklearn.metrics import classification_report, roc_auc_score
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Load dataset
dataset = EllipticBitcoinDataset(root='../data/elliptic')
data = dataset[0]

# Load timesteps
feat_df = pd.read_csv('../data/elliptic/raw/elliptic_txs_features.csv', header=None)
class_df = pd.read_csv('../data/elliptic/raw/elliptic_txs_classes.csv')
feat_df.columns = ['txId'] + [f'f{i}' for i in range(1, feat_df.shape[1])]
feat_df['timestep'] = feat_df['f1'].astype(int)
class_df.columns = ['txId', 'label']
df_meta = feat_df[['txId', 'timestep']].merge(class_df, on='txId')
timesteps = torch.tensor(df_meta['timestep'].values)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data = data.to(device)
timesteps = timesteps.to(device)

print(f"Device: {device}")
print(f"Dataset loaded — Nodes: {data.num_nodes:,} | Features: {data.num_node_features}")

Device: cuda
Dataset loaded — Nodes: 203,769 | Features: 165


## 1. Model Definition

GAT uses multi-head attention to learn different importance weights for each neighbor.
We use 4 attention heads in the first two layers, concatenating their outputs,
and a single head in the final layer.

In [2]:
class GAT(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads=4, dropout=0.5):
        super().__init__()
        self.conv1 = GATConv(in_channels, hidden_channels, heads=heads, dropout=dropout)
        self.conv2 = GATConv(hidden_channels * heads, hidden_channels, heads=heads, dropout=dropout)
        self.conv3 = GATConv(hidden_channels * heads, out_channels, heads=1, concat=False, dropout=dropout)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv1(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv3(x, edge_index)
        return x

model = GAT(in_channels=165, hidden_channels=32, out_channels=2).to(device)

print(model)
print(f"\nParameters: {sum(p.numel() for p in model.parameters()):,}")

GAT(
  (conv1): GATConv(165, 32, heads=4)
  (conv2): GATConv(128, 32, heads=4)
  (conv3): GATConv(128, 2, heads=1)
)

Parameters: 38,534


## 2. Training Setup

Same temporal split and class weighting as GCN for a fair comparison.

In [3]:
# Masks and labels
labeled_mask = data.y != 2
train_mask = labeled_mask & (timesteps <= 34)
test_mask = labeled_mask & (timesteps > 34)

y_binary = torch.zeros(data.y.shape, dtype=torch.long).to(device)
y_binary[data.y == 1] = 1
y_binary[data.y == 2] = 0

# Class weights
n_licit = (y_binary[train_mask] == 0).sum().item()
n_illicit = (y_binary[train_mask] == 1).sum().item()
weight = torch.tensor([1.0, n_licit / n_illicit]).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)
criterion = torch.nn.CrossEntropyLoss(weight=weight)

print(f"Train nodes: {train_mask.sum():,}")
print(f"Test nodes:  {test_mask.sum():,}")
print(f"Class weight illicit: {n_licit/n_illicit:.1f}x")

Train nodes: 29,894
Test nodes:  16,670
Class weight illicit: 7.6x


## 3. Training Loop

In [4]:
def train():
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = criterion(out[train_mask], y_binary[train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()

def evaluate(mask):
    model.eval()
    with torch.no_grad():
        out = model(data.x, data.edge_index)
        pred = out.argmax(dim=1)
        prob = F.softmax(out, dim=1)[:, 1]

    y_true = y_binary[mask].cpu().numpy()
    y_pred = pred[mask].cpu().numpy()
    y_prob = prob[mask].cpu().numpy()

    auc = roc_auc_score(y_true, y_prob)
    return y_true, y_pred, y_prob, auc

losses = []
test_aucs = []

for epoch in range(1, 201):
    loss = train()
    losses.append(loss)

    if epoch % 10 == 0:
        y_true, y_pred, y_prob, auc = evaluate(test_mask)
        test_aucs.append(auc)
        print(f"Epoch {epoch:3d} | Loss: {loss:.4f} | Test AUC: {auc:.4f}")

Epoch  10 | Loss: 0.9007 | Test AUC: 0.7703
Epoch  20 | Loss: 0.6781 | Test AUC: 0.8192
Epoch  30 | Loss: 0.5905 | Test AUC: 0.8113
Epoch  40 | Loss: 0.5571 | Test AUC: 0.8262
Epoch  50 | Loss: 0.5553 | Test AUC: 0.8288
Epoch  60 | Loss: 0.5315 | Test AUC: 0.8349
Epoch  70 | Loss: 0.5306 | Test AUC: 0.8392
Epoch  80 | Loss: 0.5240 | Test AUC: 0.8369
Epoch  90 | Loss: 0.5096 | Test AUC: 0.8395
Epoch 100 | Loss: 0.5118 | Test AUC: 0.8413
Epoch 110 | Loss: 0.5039 | Test AUC: 0.8426
Epoch 120 | Loss: 0.5016 | Test AUC: 0.8432
Epoch 130 | Loss: 0.4986 | Test AUC: 0.8407
Epoch 140 | Loss: 0.4924 | Test AUC: 0.8421
Epoch 150 | Loss: 0.4993 | Test AUC: 0.8437
Epoch 160 | Loss: 0.4870 | Test AUC: 0.8445
Epoch 170 | Loss: 0.4838 | Test AUC: 0.8435
Epoch 180 | Loss: 0.4872 | Test AUC: 0.8458
Epoch 190 | Loss: 0.4907 | Test AUC: 0.8460
Epoch 200 | Loss: 0.4831 | Test AUC: 0.8480


## 4. Results

In [5]:
y_true, y_pred, y_prob, auc = evaluate(test_mask)

gat_recall = (y_pred[y_true==1]==1).sum() / (y_true==1).sum() * 100
gat_precision = (y_true[y_pred==1]==1).sum() / (y_pred==1).sum() * 100

print("=== GAT Results ===")
print(classification_report(y_true, y_pred, target_names=['Licit', 'Illicit']))
print(f"ROC-AUC: {auc:.4f}")
print(f"\n--- Model Comparison ---")
print(f"{'Model':<20} {'AUC':<10} {'Illicit Recall':<20} {'Illicit Precision'}")
print(f"{'Random Forest':<20} {'0.9500':<10} {'11%':<20} {'74%'}")
print(f"{'GCN':<20} {'0.7974':<10} {'41%':<20} {'13%'}")
print(f"{'GAT':<20} {auc:<10.4f} {f'{gat_recall:.0f}%':<20} {f'{gat_precision:.0f}%'}")

=== GAT Results ===
              precision    recall  f1-score   support

       Licit       0.99      0.61      0.75     15587
     Illicit       0.14      0.91      0.24      1083

    accuracy                           0.63     16670
   macro avg       0.56      0.76      0.50     16670
weighted avg       0.93      0.63      0.72     16670

ROC-AUC: 0.8480

--- Model Comparison ---
Model                AUC        Illicit Recall       Illicit Precision
Random Forest        0.9500     11%                  74%
GCN                  0.7974     41%                  13%
GAT                  0.8480     91%                  14%
